# ⚡ WavLM Word-Level Extraction — OPTIMIZED
**Key optimization:** Pre-load all audio, batch WavLM inference
**Expected speedup:** 6-10x faster than per-video loading
**Runtime:** ~4-5 hours total (single Colab session!)

**How it works:**
1. Load WavLM on GPU (once)
2. Pre-load ALL audio files from Drive into memory
3. Batch process words through WavLM
4. Stream features to Drive

**IMPORTANT:** Audio files must be in `/content/audio/` directory
Run Cell 2 FIRST to download audio from Drive


In [ ]:
# Cell 1: Setup + Install
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, subprocess
subprocess.run(['pip', 'install', '-q', 'soundfile', 'librosa', 'transformers'], capture_output=True)
print('✓ Setup complete')

In [ ]:
# Cell 2: Download all audio from Drive
import os, subprocess, time

BASE = '/content/drive/MyDrive/standup4ai'
AUDIO_DIR = '/content/audio'
os.makedirs(AUDIO_DIR, exist_ok=True)

# Count available audio
audio_count = 0
for d in [f'{BASE}/audio', f'{BASE}/audio_1000']:
    if os.path.exists(d):
        audio_count += len([f for f in os.listdir(d) if f.endswith(('.wav', '.m4a'))])
print(f'Found {audio_count} audio files on Drive')

# Download all audio using rclone (fast)
# rclone copies directly from Drive to local
print('Downloading audio from Drive...')
t0 = time.time()

# Copy audio/ folder
if os.path.exists(f'{BASE}/audio'):
    result = subprocess.run([
        'rclone', 'copy', '-P',
        f'{BASE}/audio/',
        f'{AUDIO_DIR}/',
        '--drive-chunk-size', '64M',
        '--transfers', '8'
    ], capture_output=True, text=True)
    print(f'audio/ copied')

# Copy audio_1000/ folder
if os.path.exists(f'{BASE}/audio_1000'):
    result = subprocess.run([
        'rclone', 'copy', '-P',
        f'{BASE}/audio_1000/',
        f'{AUDIO_DIR}/',
        '--drive-chunk-size', '64M',
        '--transfers', '8'
    ], capture_output=True, text=True)
    print(f'audio_1000/ copied')

elapsed = time.time() - t0
local_count = len([f for f in os.listdir(AUDIO_DIR) if f.endswith(('.wav', '.m4a'))])
print(f'\nDownloaded {local_count} files in {elapsed/60:.0f} min')
print(f'Audio directory: {AUDIO_DIR}')

In [ ]:
# Cell 3: Load WavLM + Pre-load all audio
import torch
from transformers import AutoModel
import soundfile as sf
import numpy as np
import librosa

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'GPU: {device}')

# Load WavLM once
print('Loading WavLM...')
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device)
wavlm.eval()
print('✓ WavLM loaded')

# Pre-load all audio into memory
AUDIO_DIR = '/content/audio'
audio_files = {}

print('Pre-loading all audio into memory...')
t0 = time.time()
for i, fname in enumerate(sorted(os.listdir(AUDIO_DIR))):
    if not fname.endswith(('.wav', '.m4a')): continue
    vid = fname.rsplit('.', 1)[0]
    path = f'{AUDIO_DIR}/{fname}'
    try:
        y, sr = sf.read(path, dtype='float32')
        if len(y.shape) > 1: y = y.mean(axis=1)
        if sr != 16000: y = librosa.resample(y, orig_sr=sr, target_sr=16000)
        audio_files[vid] = y
        if (i+1) % 100 == 0:
            print(f'  Loaded {i+1} files...')
    except Exception as e:
        print(f'  Error {vid}: {e}')

elapsed = time.time() - t0
print(f'\n✓ Loaded {len(audio_files)} audio files in {elapsed/60:.0f} min')
print(f'Memory: {sum(len(y) for y in audio_files.values()) * 4 / 1e9:.1f} GB')

In [ ]:
# Cell 4: Extraction functions
SR = 16000

def extract_word_features_batch(audio, timestamps):
    """Batch extract WavLM + prosody for all words in one video.
    Audio is already loaded in memory - no I/O!"""
    n = len(timestamps)
    
    # Prepare batch: pad/trim all words to 1 second
    max_len = SR  # 1 second at 16kHz
    batch = np.zeros((n, max_len), dtype=np.float32)
    valid_mask = np.zeros(n, dtype=bool)
    
    for i, (t0, t1) in enumerate(timestamps):
        s = int(t0 * SR)
        e = min(int(t1 * SR), len(audio))
        length = e - s
        if length < 0.05 * SR: continue
        chunk = audio[s:e]
        if length >= max_len:
            chunk = chunk[:max_len]
        else:
            batch[i, :length] = chunk
        valid_mask[i] = True
    
    if valid_mask.sum() == 0:
        return np.zeros((n, 791), dtype=np.float32)
    
    # Batch WavLM inference
    valid_indices = np.where(valid_mask)[0]
    valid_chunks = batch[valid_mask]
    
    # Process in sub-batches to fit in GPU memory
    sub_batch_size = 32
    wavlm_feats = np.zeros((len(valid_indices), 768), dtype=np.float32)
    
    for i in range(0, len(valid_chunks), sub_batch_size):
        sub = torch.tensor(valid_chunks[i:i+sub_batch_size]).unsqueeze(1).to(device)  # (sub_batch, 1, max_len)
        with torch.no_grad():
            out = wavlm(sub)
        wavlm_feats[i:i+sub_batch_size] = out.last_hidden_state.mean(dim=2).squeeze(1).cpu().numpy()
    
    # Extract prosody for valid words
    prosody_feats = np.zeros((len(valid_indices), 23), dtype=np.float32)
    y_22k = librosa.resample(audio, orig_sr=16000, target_sr=22050)
    
    for j, idx in enumerate(valid_indices):
        t0, t1 = timestamps[idx]
        s = int(t0 * 22050)
        e = min(int(t1 * 22050), len(y_22k))
        chunk = y_22k[s:e]
        if len(chunk) < 0.05 * 22050:
            continue
        # F0 (5 dims)
        try:
            f0, voiced, _ = librosa.pyin(chunk, fmin=50, fmax=500, sr=22050)
            f0c = f0[~np.isnan(f0)]
            v = voiced[~np.isnan(f0)]
            f0_f = [np.mean(f0c), np.std(f0c), np.max(f0c), np.min(f0c), np.mean(v)] if len(f0c) > 0 else [0]*5
        except: f0_f = [0]*5
        # Energy (5 dims)
        hop = 512
        rms = librosa.feature.rms(y=chunk, hop_length=hop)[0]
        energy_f = [np.mean(rms), np.std(rms), np.max(rms), np.min(rms), np.max(rms)-np.min(rms)]
        # Duration (2 dims)
        dur = len(chunk) / 22050
        dur_f = [dur, dur / (np.sum(rms > np.mean(rms)) + 1)]
        # Spectral (5 dims)
        try:
            sc = librosa.feature.spectral_centroid(y=chunk, sr=22050, hop_length=hop)[0]
            sb = librosa.feature.spectral_bandwidth(y=chunk, sr=22050, hop_length=hop)[0]
            sf2 = librosa.feature.spectral_flatness(y=chunk, hop_length=hop)[0]
            zcr = librosa.feature.zero_crossing_rate(chunk, hop_length=hop)[0]
            spec_f = [np.mean(sc), np.mean(sb), np.mean(sf2), np.mean(zcr), np.std(zcr)]
        except: spec_f = [0]*5
        # Voice quality (6 dims)
        try:
            yh, _ = librosa.effects.hpss(chunk)
            hnr = np.mean(np.abs(yh)) / (np.mean(np.abs(chunk)) + 1e-8)
            voice_f = [hnr, np.mean(np.abs(chunk)), np.std(chunk), np.max(np.abs(chunk)), 0, 0]
        except: voice_f = [0]*6
        prosody_feats[j] = np.array(f0_f + energy_f + dur_f + spec_f + voice_f, dtype=np.float32)
    
    # Combine into full feature matrix
    full_feats = np.zeros((n, 791), dtype=np.float32)
    full_feats[valid_indices, :768] = wavlm_feats
    full_feats[valid_indices, 768:] = prosody_feats
    
    return full_feats

print('✓ Extraction functions ready')

In [ ]:
# Cell 5: Find videos + extraction loop
import json, time, ast, pandas as pd

BASE = '/content/drive/MyDrive/standup4ai'
FEAT_DIR = f'{BASE}/word_level_features_opt'
LABEL_BASE = f'{BASE}/seq-Standup4AI/dataset'
os.makedirs(FEAT_DIR, exist_ok=True)

# Find all videos with labels
all_videos = []
for lang in ['en_uk', 'en_us', 'es', 'es_latam', 'fr', 'fr_ca', 'it', 'cs', 'hu', 'pt']:
    label_dir = f'{LABEL_BASE}/{lang}/emnlp+jahak/all'
    if not os.path.exists(label_dir): continue
    for fname in sorted(os.listdir(label_dir)):
        if not fname.endswith('.csv'): continue
        vid = fname.replace('.csv', '')
        all_videos.append((vid, lang))

print(f'Total videos with labels: {len(all_videos)}')

# Load checkpoint
ckpt_file = f'{FEAT_DIR}/checkpoint.json'
if os.path.exists(ckpt_file):
    with open(ckpt_file) as f:
        ckpt = json.load(f)
    done_vids = set(ckpt.get('done', []))
    print(f'Already done: {len(done_vids)}')
else:
    done_vids = set()
    ckpt = {'done': []}

pending = [(v, l) for v, l in all_videos if v not in done_vids]
print(f'Pending: {len(pending)}')

# Extraction loop
t0 = time.time()
SAVE_EVERY = 20  # checkpoint every 20 videos

for i, (vid, lang) in enumerate(pending):
    feat_path = f'{FEAT_DIR}/{vid}_features.npy'
    if os.path.exists(feat_path): continue
    
    # Check if audio in memory
    if vid not in audio_files:
        # Try to find audio
        audio_path = None
        for d in [f'{BASE}/audio', f'{BASE}/audio_1000', '/content/audio']:
            for ext in ['.wav', '.m4a']:
                p = f'{d}/{vid}{ext}'
                if os.path.exists(p):
                    try:
                        y, sr = sf.read(p, dtype='float32')
                        if len(y.shape) > 1: y = y.mean(axis=1)
                        if sr != 16000: y = librosa.resample(y, orig_sr=sr, target_sr=16000)
                        audio_files[vid] = y
                        audio_path = p
                        break
                    except: pass
        if vid not in audio_files:
            continue  # Skip if no audio
    
    audio = audio_files[vid]
    label_path = f'{LABEL_BASE}/{lang}/emnlp+jahak/all/{vid}.csv'
    
    try:
        df = pd.read_csv(label_path)
        timestamps = []
        for _, row in df.iterrows():
            try:
                ts = ast.literal_eval(str(row['timestamp']))
                timestamps.append((float(ts[0]), float(ts[1]))
            except: pass
        
        features = extract_word_features_batch(audio, timestamps)
        np.save(feat_path, features)
        
        labels = np.array([1 if str(r.get('label','O')).strip() in ('B','I','L') else 0
                       for _, r in df.iterrows()], dtype=np.int32)[:len(features)]
        np.save(f'{FEAT_DIR}/{vid}_labels.npy', labels)
        
        done_vids.add(vid)
        
        elapsed = time.time() - t0
        rate = (i + 1) / (elapsed / 60)  # videos per minute
        eta_min = (len(pending) - i - 1) / rate if rate > 0 else 0
        
        print(f'[{i+1}/{len(pending)}] {vid}: {len(timestamps)} words | '
              f'Done: {len(done_vids)} | ETA: {eta_min:.0f} min | {rate:.1f} vid/min', flush=True)
        
        if (i + 1) % SAVE_EVERY == 0:
            ckpt['done'] = list(done_vids)
            with open(ckpt_file, 'w') as f:
                json.dump(ckpt, f)
            print(f'  Checkpoint saved ({len(done_vids)} done)')
            
    except Exception as e:
        print(f'ERROR {vid}: {str(e)[:80]}')

# Final save
ckpt['done'] = list(done_vids)
with open(ckpt_file, 'w') as f:
    json.dump(ckpt, f)
print(f'\n✓ COMPLETE: {len(done_vids)} videos extracted')
print(f'Total time: {(time.time()-t0)/3600:.1f} hours')

In [ ]:
# Cell 6: Summary
print('=== EXTRACTION COMPLETE ===')
done_vids = ckpt['done']
print(f'Total: {len(done_vids)} videos')

# Check features
feature_files = [f for f in os.listdir(FEAT_DIR) if f.endswith('_features.npy')]
print(f'Feature files: {len(feature_files)}')

# Sample
for f in feature_files[:3]:
    vid = f.replace('_features.npy', '')
    feat = np.load(f'{FEAT_DIR}/{f}')
    labels = np.load(f'{FEAT_DIR}/{vid}_labels.npy') if os.path.exists(f'{FEAT_DIR}/{vid}_labels.npy') else None
    print(f'  {vid}: {feat.shape}, {len(labels) if labels is not None else "?"} labels')

print(f'\nFeatures saved to: {FEAT_DIR}')